# INSTRUCTOR SOLUTION: Machine Translation
## AIAT 122 - Deep Learning

**⚠️ INSTRUCTOR USE ONLY**

Complete transformer solution for translation.

## 📥 Inputs & 📤 Outputs | المدخلات والمخرجات

**Inputs:** What we use in this notebook

- Libraries and concepts as introduced in this notebook; see prerequisites and code comments.

**Outputs:** What you'll see when you run the cells

- Printed results, figures, and summaries as shown when you run the cells.

---


In [ ]:
# INSTRUCTOR SOLUTION: Transformer Seq2Seq (pure PyTorch)
# ─────────────────────────────────────────────────────────
import torch, torch.nn as nn, torch.optim as optim, numpy as np
print(f'PyTorch {torch.__version__}')
print('✅ Seq2seq Transformer loaded (pure PyTorch)')

# ── Toy reversal task ────────────────────────────────────────────────────
torch.manual_seed(0); np.random.seed(0)
SOS, EOS, PAD, V = 10, 11, 12, 13

def make_pair(n=8):
    s = np.random.randint(0, 10, n).tolist()
    return s, list(reversed(s))

def collate(pairs):
    # tgt_in  = [SOS, y1, y2, ..., yn]   — length n+1
    # tgt_out = [y1,  y2, ..., yn, EOS]  — length n+1
    src     = torch.tensor([p[0] for p in pairs], dtype=torch.long)
    tgt_in  = torch.tensor([[SOS] + p[1] for p in pairs], dtype=torch.long)
    tgt_out = torch.tensor([p[1] + [EOS] for p in pairs], dtype=torch.long)
    return src, tgt_in, tgt_out

class Seq2Seq(nn.Module):
    def __init__(self, V, d=64, nhead=4, nlayers=2):
        super().__init__()
        self.se = nn.Embedding(V, d); self.te = nn.Embedding(V, d)
        self.transformer = nn.Transformer(d, nhead, nlayers, nlayers,
                                          dim_feedforward=128, batch_first=True)
        self.fc = nn.Linear(d, V)
    def forward(self, src, tgt):
        mask = nn.Transformer.generate_square_subsequent_mask(tgt.size(1))
        out  = self.transformer(self.se(src), self.te(tgt),
                                tgt_mask=mask, tgt_is_causal=True)
        return self.fc(out)

model   = Seq2Seq(V)
opt     = optim.Adam(model.parameters(), lr=3e-3)
loss_fn = nn.CrossEntropyLoss(ignore_index=PAD)
pairs   = [make_pair() for _ in range(8000)]

print('Training (20 epochs) …')
for epoch in range(20):
    model.train(); el = 0
    for i in range(0, len(pairs), 256):
        src, ti, to = collate(pairs[i:i+256])
        logits = model(src, ti)              # (B, T, V)
        loss   = loss_fn(logits.reshape(-1, V), to.reshape(-1))
        opt.zero_grad(); loss.backward(); opt.step(); el += loss.item()
    if (epoch+1) % 5 == 0:
        print(f'  Epoch {epoch+1}: loss={el:.4f}')

def decode(src_seq, max_len=12):
    model.eval()
    src = torch.tensor([src_seq], dtype=torch.long)
    tgt = torch.tensor([[SOS]], dtype=torch.long)
    for _ in range(max_len):
        with torch.no_grad(): logits = model(src, tgt)
        nxt = logits[0, -1].argmax().item()
        if nxt == EOS: break
        tgt = torch.cat([tgt, torch.tensor([[nxt]])], dim=1)
    return tgt[0, 1:].tolist()

print('\nTask 1 (Reversal) — 10 test samples:')
test = [make_pair() for _ in range(10)]
ok = sum(decode(s) == t for s, t in test)
for s, t in test[:5]:
    print(f'  {s} → expected {t}, got {decode(s)}')
print(f'Accuracy: {ok}/10')
print('\nTeaching Notes: Grading  Task 1 (35pts) Task 2 (30pts) Task 3 (35pts)')